# 📖 Notebook 3: Consistent Hashing for Cache Routing

In Notebook 1 we saw that **modulo hashing** breaks when nodes are added or removed —
~75% of keys need to move. **Consistent hashing** is the industry-standard solution.

The big idea: map both **nodes** and **keys** onto a circular ring. Each key is stored
on the next node clockwise from its position. When a node joins or leaves, only the
keys between it and its neighbor need to move — roughly **1/N** of all keys.

## Learning Objectives

By the end of this notebook, you'll be able to:
- Build a consistent hash ring from scratch
- Understand why virtual nodes are needed for even distribution
- Measure key movement when nodes join or leave
- Use consistent hashing to route reads/writes to a real Redis cluster
- Visualize the hash ring and key distribution

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/distributed-cache
docker-compose up -d
```

### RedisInsight (Redis GUI)
- **URL**: http://localhost:5540
- Add each node: `localhost:6381`, `localhost:6382`, `localhost:6383`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import redis
import hashlib
import bisect
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Our 3-node cluster
REDIS_NODES = {
    "node-1": redis.Redis(host="localhost", port=6381, decode_responses=True),
    "node-2": redis.Redis(host="localhost", port=6382, decode_responses=True),
    "node-3": redis.Redis(host="localhost", port=6383, decode_responses=True),
}

for name, client in REDIS_NODES.items():
    client.flushall()
    try:
        client.ping()
        print(f"✅ {name} is up")
    except Exception as e:
        print(f"❌ {name} failed: {e}")

RING_SIZE = 2**32  # standard 32-bit hash space


def hash_key(key: str) -> int:
    """Hash a string to a position on the ring (0 to 2^32 - 1)."""
    return int(hashlib.md5(key.encode()).hexdigest(), 16) % RING_SIZE

---

## Part 1: The Hash Ring (No Virtual Nodes)

The simplest version of consistent hashing:

1. Hash each node name to get its position on a ring (0 to 2³²)
2. Hash each key to get its position
3. Walk clockwise from the key's position until you hit a node
4. That node stores the key

```
        node-1
       /      \
      /   key   \
     |    ↓      |
     |   ●→→→ node-2   ← key goes here (next clockwise)
      \        /
       \      /
        node-3
```

In [ ]:
class ConsistentHashRing:
    """
    A consistent hash ring that maps keys to nodes.

    Each physical node gets `virtual_nodes` positions on the ring.
    More virtual nodes = more even distribution.
    """

    def __init__(self, virtual_nodes: int = 1):
        self.virtual_nodes = virtual_nodes
        self.ring = {}         # position → node_name
        self.sorted_positions = []  # sorted list of positions (for bisect)
        self.physical_nodes = set()

    def add_node(self, node_name: str):
        """Add a node to the ring with `virtual_nodes` positions."""
        self.physical_nodes.add(node_name)
        for i in range(self.virtual_nodes):
            # Each virtual node gets a unique key: "node-1#0", "node-1#1", etc.
            virtual_key = f"{node_name}#{i}"
            position = hash_key(virtual_key)
            self.ring[position] = node_name
            bisect.insort(self.sorted_positions, position)

    def remove_node(self, node_name: str):
        """Remove a node and all its virtual positions from the ring."""
        self.physical_nodes.discard(node_name)
        positions_to_remove = [
            pos for pos, name in self.ring.items() if name == node_name
        ]
        for pos in positions_to_remove:
            del self.ring[pos]
            self.sorted_positions.remove(pos)

    def get_node(self, key: str) -> str:
        """Find which node should store a given key."""
        if not self.ring:
            raise ValueError("No nodes in the ring!")

        position = hash_key(key)

        # Find the first node position >= key position (clockwise)
        idx = bisect.bisect_right(self.sorted_positions, position)

        # If we've gone past the end, wrap around to the first node
        if idx >= len(self.sorted_positions):
            idx = 0

        node_position = self.sorted_positions[idx]
        return self.ring[node_position]

    def get_distribution(self, keys: list[str]) -> dict[str, int]:
        """Count how many keys each node gets."""
        counts = {node: 0 for node in self.physical_nodes}
        for key in keys:
            node = self.get_node(key)
            counts[node] += 1
        return counts


# Build a ring with 1 virtual node per physical node (simplest case)
ring = ConsistentHashRing(virtual_nodes=1)
for name in ["node-1", "node-2", "node-3"]:
    ring.add_node(name)

# Show node positions on the ring
print("🔵 Node positions on the hash ring (1 virtual node each):")
print(f"   Ring size: 0 to {RING_SIZE:,} (2³²)")
print()
for pos in ring.sorted_positions:
    node = ring.ring[pos]
    pct = pos / RING_SIZE * 100
    print(f"   Position {pos:>12,} ({pct:>5.1f}%) → {node}")

# Route some keys
print()
print("🗂️  Key routing:")
for i in range(8):
    key = f"product:{i}"
    node = ring.get_node(key)
    pos = hash_key(key)
    print(f"   {key:<15} position={pos:>12,} → {node}")

### ⚠️ The Problem: Uneven Distribution

With only 1 position per node, the ring is unevenly divided. Some nodes get way
more keys than others. This is like cutting a pizza with random slices — some
pieces will be huge and others tiny.

In [ ]:
# Check distribution with 1 virtual node
test_keys = [f"key:{i}" for i in range(10000)]
dist = ring.get_distribution(test_keys)

ideal = len(test_keys) / len(dist)

print("📊 Key distribution with 1 virtual node per physical node (10,000 keys):")
print(f"   Ideal: {ideal:.0f} keys per node")
print()
for node, count in sorted(dist.items()):
    deviation = (count - ideal) / ideal * 100
    bar = "█" * (count // 100)
    status = "🔴" if abs(deviation) > 30 else "🟡" if abs(deviation) > 10 else "🟢"
    print(f"   {node}: {count:>5} keys ({deviation:>+6.1f}% from ideal) {status}  {bar}")

print()
print("⚠️  With few virtual nodes, distribution is often very uneven.")
print("   One node might handle 2-3× more keys than another.")
print("   Solution: add more virtual nodes per physical node.")

---

## Part 2: Virtual Nodes Fix the Balance

Instead of placing each node at 1 position on the ring, we place it at **many**
positions (virtual nodes). This breaks the ring into smaller, more uniform segments.

Think of it like this:
- 1 virtual node = 1 point on the ring → very uneven
- 10 virtual nodes = 10 points per node → much better
- 150 virtual nodes = 150 points per node → nearly perfect

More virtual nodes = more even distribution, but slightly more memory for the ring.

In [ ]:
# Compare distribution with different numbers of virtual nodes

test_keys = [f"key:{i}" for i in range(10000)]
nodes = ["node-1", "node-2", "node-3"]
ideal = len(test_keys) / len(nodes)

print("📊 How virtual nodes improve balance (10,000 keys, 3 physical nodes)")
print("=" * 70)
print(f"{'Virtual Nodes':>14} {'node-1':>8} {'node-2':>8} {'node-3':>8} {'Max Deviation':>15}")
print("-" * 70)

for vn in [1, 5, 10, 25, 50, 100, 150, 200]:
    ring = ConsistentHashRing(virtual_nodes=vn)
    for name in nodes:
        ring.add_node(name)

    dist = ring.get_distribution(test_keys)
    counts = [dist[n] for n in nodes]
    max_dev = max(abs(c - ideal) / ideal * 100 for c in counts)

    status = "🟢" if max_dev < 5 else "🟡" if max_dev < 15 else "🔴"
    print(f"  {vn:>12}   {counts[0]:>6}   {counts[1]:>6}   {counts[2]:>6}   {max_dev:>10.1f}%  {status}")

print()
print("🔑 With ~150 virtual nodes, deviation is typically under 5%.")
print("   This is what production systems like Amazon DynamoDB use.")

In [ ]:
# Visualize the hash ring with virtual nodes

def visualize_ring(ring: ConsistentHashRing, title: str, keys: list[str] = None):
    """Draw the hash ring showing node positions and optionally key positions."""
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))

    # Draw the ring
    theta = np.linspace(0, 2 * np.pi, 200)
    ax.plot(np.cos(theta), np.sin(theta), 'lightgray', linewidth=2)

    # Color map for nodes
    colors = {'node-1': '#FF6B6B', 'node-2': '#4ECDC4', 'node-3': '#45B7D1',
              'node-4': '#96CEB4', 'node-5': '#FFEAA7'}

    # Plot node positions on the ring
    for pos in ring.sorted_positions:
        node = ring.ring[pos]
        angle = 2 * math.pi * pos / RING_SIZE
        x, y = math.cos(angle), math.sin(angle)
        color = colors.get(node, 'gray')
        ax.plot(x, y, 'o', color=color, markersize=10, markeredgecolor='black',
                markeredgewidth=0.5)

    # Plot key positions (smaller dots)
    if keys:
        for key in keys[:50]:  # limit to 50 keys for readability
            pos = hash_key(key)
            node = ring.get_node(key)
            angle = 2 * math.pi * pos / RING_SIZE
            x, y = math.cos(angle), math.sin(angle)
            color = colors.get(node, 'gray')
            ax.plot(x * 0.95, y * 0.95, '.', color=color, markersize=4, alpha=0.6)

    # Legend
    legend_handles = [
        mpatches.Patch(color=colors.get(n, 'gray'), label=n)
        for n in sorted(ring.physical_nodes)
    ]
    ax.legend(handles=legend_handles, loc='upper right', fontsize=10)

    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-1.3, 1.3)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.show()


# Ring with 1 virtual node
ring_1 = ConsistentHashRing(virtual_nodes=1)
for n in ["node-1", "node-2", "node-3"]:
    ring_1.add_node(n)

sample_keys = [f"product:{i}" for i in range(50)]
visualize_ring(ring_1, "Hash Ring — 1 Virtual Node per Physical Node", sample_keys)

# Ring with 20 virtual nodes
ring_20 = ConsistentHashRing(virtual_nodes=20)
for n in ["node-1", "node-2", "node-3"]:
    ring_20.add_node(n)

visualize_ring(ring_20, "Hash Ring — 20 Virtual Nodes per Physical Node", sample_keys)

---

## Part 3: Adding & Removing Nodes (Minimal Key Movement)

The whole point of consistent hashing: when a node joins or leaves, only a small
fraction of keys need to move. Let's prove it.

In [ ]:
# Measure key movement when adding a node

VN = 150  # virtual nodes — production-like
test_keys = [f"key:{i}" for i in range(100000)]

# Start with 3 nodes
ring_before = ConsistentHashRing(virtual_nodes=VN)
for n in ["node-1", "node-2", "node-3"]:
    ring_before.add_node(n)

# Record where each key goes
assignment_before = {key: ring_before.get_node(key) for key in test_keys}

# Add a 4th node
ring_after = ConsistentHashRing(virtual_nodes=VN)
for n in ["node-1", "node-2", "node-3", "node-4"]:
    ring_after.add_node(n)

assignment_after = {key: ring_after.get_node(key) for key in test_keys}

# Count movements
moved = sum(1 for k in test_keys if assignment_before[k] != assignment_after[k])
ideal_moved = len(test_keys) / 4  # theoretical: 1/N keys should move

print("📊 Key movement when adding node-4 (3 → 4 nodes)")
print(f"   Total keys:     {len(test_keys):>8,}")
print(f"   Keys moved:     {moved:>8,} ({moved/len(test_keys)*100:.1f}%)")
print(f"   Ideal (1/N):    {ideal_moved:>8,.0f} ({100/4:.1f}%)")
print()

# Compare with modulo hashing
modulo_moved = sum(
    1 for key in test_keys
    if (hash_key(key) % 3) != (hash_key(key) % 4)
)

print("📊 Comparison:")
print(f"   Consistent hashing: {moved:>8,} keys moved ({moved/len(test_keys)*100:.1f}%)  🟢")
print(f"   Modulo hashing:     {modulo_moved:>8,} keys moved ({modulo_moved/len(test_keys)*100:.1f}%)  🔴")
print()
print(f"   Consistent hashing moved {modulo_moved/max(moved,1):.0f}× fewer keys!")

In [ ]:
# Simulate scaling from 3 to 10 nodes, one at a time

VN = 150
test_keys = [f"key:{i}" for i in range(100000)]

print("📊 Key movement as we scale from 3 to 10 nodes (consistent hashing)")
print("=" * 65)
print(f"{'Change':<15} {'Keys Moved':>12} {'Percentage':>12} {'Ideal (1/N)':>12}")
print("-" * 65)

prev_ring = ConsistentHashRing(virtual_nodes=VN)
for n in ["node-1", "node-2", "node-3"]:
    prev_ring.add_node(n)
prev_assignment = {key: prev_ring.get_node(key) for key in test_keys}

for new_node_num in range(4, 11):
    new_ring = ConsistentHashRing(virtual_nodes=VN)
    for i in range(1, new_node_num + 1):
        new_ring.add_node(f"node-{i}")

    new_assignment = {key: new_ring.get_node(key) for key in test_keys}
    moved = sum(1 for k in test_keys if prev_assignment[k] != new_assignment[k])
    pct = moved / len(test_keys) * 100
    ideal_pct = 100 / new_node_num

    print(f"  {new_node_num-1}→{new_node_num} nodes     {moved:>8,}     {pct:>8.1f}%     {ideal_pct:>8.1f}%")

    prev_assignment = new_assignment

print()
print("🔑 Each time we add a node, only ~1/N keys move.")
print("   This means scaling is SMOOTH — no thundering herd to the database.")

In [ ]:
# What about removing a node? (simulating a crash)

VN = 150
test_keys = [f"key:{i}" for i in range(100000)]

# Start with 5 nodes
ring_5 = ConsistentHashRing(virtual_nodes=VN)
for i in range(1, 6):
    ring_5.add_node(f"node-{i}")

assignment_5 = {key: ring_5.get_node(key) for key in test_keys}

# Remove node-3 (simulating a crash)
ring_4 = ConsistentHashRing(virtual_nodes=VN)
for i in [1, 2, 4, 5]:  # skip node-3
    ring_4.add_node(f"node-{i}")

assignment_4 = {key: ring_4.get_node(key) for key in test_keys}

moved = sum(1 for k in test_keys if assignment_5[k] != assignment_4[k])
# Keys that WERE on node-3 must move (they're the only ones affected)
was_on_node3 = sum(1 for k in test_keys if assignment_5[k] == "node-3")

print("💥 Simulating node-3 crash (5 → 4 nodes)")
print(f"   Keys that were on node-3:  {was_on_node3:>8,}")
print(f"   Keys that moved:           {moved:>8,}")
print(f"   Keys that stayed put:      {len(test_keys) - moved:>8,}")
print()
print("🔑 Only the keys from the crashed node need to be reassigned.")
print("   The other ~80% of keys are completely unaffected!")

# Show where node-3's keys went
print()
print("📊 Where did node-3's keys go?")
redistribution = {}
for k in test_keys:
    if assignment_5[k] == "node-3":
        new_node = assignment_4[k]
        redistribution[new_node] = redistribution.get(new_node, 0) + 1

for node, count in sorted(redistribution.items()):
    bar = "█" * (count // 100)
    print(f"   → {node}: {count:>6} keys  {bar}")

print()
print("   Keys from the failed node are spread across remaining nodes.")

---

## Part 4: Using Consistent Hashing with Real Redis Nodes

Let's put it all together — use our consistent hash ring to route real
reads and writes to our 3 Redis nodes.

In [ ]:
class DistributedCache:
    """
    A distributed cache that uses consistent hashing to route
    keys to the correct Redis node.

    This is what a real cache client (like the Redis Cluster client)
    does under the hood.
    """

    def __init__(self, nodes: dict[str, redis.Redis], virtual_nodes: int = 150):
        self.nodes = nodes
        self.ring = ConsistentHashRing(virtual_nodes=virtual_nodes)
        for name in nodes:
            self.ring.add_node(name)

    def _get_client(self, key: str) -> tuple[str, redis.Redis]:
        """Find the right Redis node for a given key."""
        node_name = self.ring.get_node(key)
        return node_name, self.nodes[node_name]

    def set(self, key: str, value: str, ttl: int = None) -> str:
        """Store a key-value pair on the appropriate node."""
        node_name, client = self._get_client(key)
        if ttl:
            client.set(key, value, ex=ttl)
        else:
            client.set(key, value)
        return node_name

    def get(self, key: str) -> tuple[str, str | None]:
        """Retrieve a value, returning (node_name, value)."""
        node_name, client = self._get_client(key)
        return node_name, client.get(key)

    def delete(self, key: str) -> str:
        """Delete a key from the appropriate node."""
        node_name, client = self._get_client(key)
        client.delete(key)
        return node_name


# Flush all nodes
for client in REDIS_NODES.values():
    client.flushall()

# Create the distributed cache
dcache = DistributedCache(REDIS_NODES, virtual_nodes=150)

# Store 1000 products
print("📦 Storing 1000 products using consistent hashing...")
for i in range(1000):
    key = f"product:{i}"
    value = f'{{"id": {i}, "name": "Product {i}", "price": {i * 1.5:.2f}}}'
    dcache.set(key, value)

# Check distribution
print()
print("📊 Actual key distribution across Redis nodes:")
for name, client in REDIS_NODES.items():
    count = client.dbsize()
    bar = "█" * (count // 10)
    print(f"   {name}: {count:>4} keys  {bar}")

# Verify reads work
print()
print("🔍 Sample reads:")
for key in ["product:0", "product:42", "product:999"]:
    node, value = dcache.get(key)
    print(f"   {key} → {node} → {value}")

print()
print("✅ Consistent hashing routes each key to the same node every time.")
print("   Open RedisInsight to see the actual keys on each node!")

---

## Part 5: Hot Key Mitigation with Key Copies

Sometimes one key gets way more traffic than others (a viral product, a celebrity
profile). Consistent hashing puts it on ONE node, which becomes a hot spot.

The fix: create **copies** of the hot key with different suffixes so they land on
different nodes. Reads pick a random copy; writes update all copies.

In [ ]:
import random

# Without hot key mitigation: all reads go to one node
hot_key = "product:iphone"
node, _ = dcache.get(hot_key)
print(f"🔥 Hot key '{hot_key}' lives on: {node}")
print(f"   ALL reads for this key hit {node} — it becomes a bottleneck!")
print()

# With hot key copies: spread reads across nodes
NUM_COPIES = 5

def set_hot_key(dcache: DistributedCache, key: str, value: str, copies: int):
    """Write a hot key to multiple nodes using random suffixes."""
    nodes_used = set()
    for i in range(copies):
        copy_key = f"{key}#copy:{i}"
        node = dcache.set(copy_key, value)
        nodes_used.add(node)
    return nodes_used


def get_hot_key(dcache: DistributedCache, key: str, copies: int) -> tuple[str, str]:
    """Read a hot key from a random copy (load-balanced)."""
    copy_idx = random.randint(0, copies - 1)
    copy_key = f"{key}#copy:{copy_idx}"
    return dcache.get(copy_key)


# Store the hot key with 5 copies
value = '{"name": "iPhone 16", "price": 999.99}'
nodes_used = set_hot_key(dcache, hot_key, value, NUM_COPIES)

print(f"📦 Hot key stored with {NUM_COPIES} copies across nodes: {nodes_used}")
print()

# Simulate 1000 reads — which nodes handle them?
read_distribution = {}
for _ in range(1000):
    node, val = get_hot_key(dcache, hot_key, NUM_COPIES)
    read_distribution[node] = read_distribution.get(node, 0) + 1

print("📊 Read distribution for hot key (1000 reads with copies):")
for node, count in sorted(read_distribution.items()):
    bar = "█" * (count // 20)
    print(f"   {node}: {count:>4} reads  {bar}")

print()
print("🔑 Without copies: 1 node handles ALL 1000 reads.")
print(f"   With {NUM_COPIES} copies: reads are spread across {len(read_distribution)} nodes!")
print("   This is how production systems handle viral content.")

---

## 🧪 Try It Yourself

1. **Vary virtual nodes**: Try 1, 10, 50, 200 virtual nodes and check the distribution.
   At what point does it become "good enough"?

2. **Simulate a node crash**: Remove a node from the ring and see which keys move.
   Verify that keys on the other nodes are unaffected.

3. **Scale up gradually**: Add nodes one at a time (3 → 4 → 5 → ... → 10) and
   track total key movement. Is it O(K/N) each time?

4. **Use different hash functions**: Replace MD5 with SHA-256 or MurmurHash (mmh3
   is in requirements.txt). Does it change the distribution quality?

5. **Open RedisInsight** and watch keys redistribute as you run the notebooks.

## 📝 Key Takeaways

| Concept | Key Insight |
|---------|-------------|
| Hash Ring | Both nodes and keys are mapped to a circular space |
| Key Routing | Walk clockwise from key position to find its node |
| Virtual Nodes | Multiple positions per node for even distribution |
| Adding a Node | Only ~1/N keys need to move (vs ~75% with modulo) |
| Removing a Node | Only the crashed node's keys are redistributed |
| Hot Key Copies | Replicate hot keys with suffixes to spread load |

## 🎓 What's Next?

You now understand the three pillars of distributed caching:
1. **Partitioning** — how to split data across nodes
2. **Coherence** — how to keep copies in sync
3. **Consistent Hashing** — how to route requests efficiently

These concepts appear in almost every distributed system:
Amazon DynamoDB, Apache Cassandra, Memcached, Redis Cluster, and more.

In [ ]:
# Cleanup
for client in REDIS_NODES.values():
    client.flushall()
print("🧹 All nodes flushed.")